# GPU Portfolio Bench — Demo Notebook

End-to-end walkthrough:
1. Fetch price data
2. CPU vs GPU Monte Carlo VaR — side-by-side timing
3. Efficient frontier (Markowitz)
4. LSTM forecasting model — quick train + inference
5. Custom CUDA kernel comparison

Run on Google Colab with a T4/A100 GPU for best results.

In [ ]:
# Clone & install (uncomment when running on Colab)
# !git clone https://github.com/scotthnguyen/gpu-portfolio-bench.git
# %cd gpu-portfolio-bench
# !pip install -q -r requirements.txt
# !pip install -q cupy-cuda12x  # match your Colab CUDA version

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data pipeline

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from src.data.fetch import fetch_prices, compute_log_returns

prices = fetch_prices()
returns = compute_log_returns(prices).dropna(axis=1)
print(f"{returns.shape[1]} assets × {returns.shape[0]} trading days")
returns.tail(3)

## 2. CPU vs GPU Monte Carlo VaR

In [ ]:
from src.models.var_cpu import compute_var_cvar_cpu
from src.models.var_gpu import compute_var_cvar_gpu

N_ASSETS = 50
N_PATHS = 1_000_000

assets = list(returns.columns[:N_ASSETS])
R = returns[assets].to_numpy(dtype=np.float64)
weights = np.ones(N_ASSETS) / N_ASSETS

cpu_r = compute_var_cvar_cpu(R, weights, n_paths=N_PATHS)
print(f"CPU  VaR={cpu_r['VaR']:.4f}  elapsed={cpu_r['elapsed_sec']:.2f}s  "
      f"throughput={cpu_r['throughput_paths_per_sec']:,.0f} paths/s")

if torch.cuda.is_available():
    gpu_r = compute_var_cvar_gpu(R, weights, n_paths=N_PATHS, device='cuda')
    speedup = cpu_r['elapsed_sec'] / gpu_r['elapsed_sec']
    print(f"GPU  VaR={gpu_r['VaR']:.4f}  elapsed={gpu_r['elapsed_sec']:.2f}s  "
          f"throughput={gpu_r['throughput_paths_per_sec']:,.0f} paths/s")
    print(f"\n→ GPU speedup: {speedup:.1f}×  |  GPU memory: {gpu_r['gpu_mem_mb']:.1f} MB")

## 3. Efficient frontier

In [ ]:
import plotly.graph_objects as go
from src.models.portfolio_opt import build_efficient_frontier

N_OPT = 20  # keep small for notebook speed
ef = build_efficient_frontier(
    returns[assets[:N_OPT]].to_numpy(dtype=np.float64),
    assets[:N_OPT],
    n_points=30,
    use_gpu=torch.cuda.is_available(),
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ef.volatilities, y=ef.expected_returns,
    mode='lines+markers',
    marker=dict(color=ef.sharpe_ratios, colorscale='Viridis', showscale=True,
                colorbar=dict(title='Sharpe')),
    text=[f'Sharpe: {s:.2f}' for s in ef.sharpe_ratios],
    name='Efficient frontier',
))
fig.update_layout(title='Efficient Frontier', xaxis_title='Annualized Volatility',
                  yaxis_title='Annualized Return')
fig.show()

## 4. LSTM return forecaster

In [ ]:
import plotly.express as px
import pandas as pd
from src.models.forecaster import train, load_model, predict_next_day

N_FORECAST_ASSETS = 10
forecast_assets = assets[:N_FORECAST_ASSETS]
R_fore = returns[forecast_assets].to_numpy(dtype=np.float32)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training LSTM on {N_FORECAST_ASSETS} assets, device={device}")

result = train(R_fore, forecast_assets, epochs=30, hidden=64, device=device)

losses = pd.DataFrame({'train': result.train_losses, 'val': result.val_losses})
px.line(losses, title='Training curves').show()

model, ckpt = load_model(result.checkpoint_path, device=device)
pred = predict_next_day(model, R_fore[-20:], device=device)

pred_df = pd.DataFrame({'Asset': forecast_assets, 'Predicted daily return': pred,
                        'Annualized (%)': pred * 252 * 100})
pred_df.sort_values('Predicted daily return', ascending=False)

## 5. Custom CUDA kernel (CuPy raw kernel)

In [ ]:
try:
    from src.models.var_cuda_kernel import compute_var_cvar_cupy
    cupy_r = compute_var_cvar_cupy(R, weights, n_paths=N_PATHS)
    speedup_vs_cpu = cpu_r['elapsed_sec'] / cupy_r['elapsed_sec']
    print(f"CuPy kernel  VaR={cupy_r['VaR']:.4f}  elapsed={cupy_r['elapsed_sec']:.2f}s")
    print(f"Speedup vs CPU: {speedup_vs_cpu:.1f}×")
    print(f"GPU memory: {cupy_r['gpu_mem_mb']:.1f} MB")
except RuntimeError as e:
    print(f"CuPy not available: {e}")

## 6. Run the full benchmark sweep

This writes `results/benchmark_sweep.csv` — feed it into the Streamlit dashboard.

```bash
python -m src.bench.run_sweep --device all
streamlit run src/dashboard/app.py
```